In [1]:
# Projet SIEM : Suricata, Filebeat, Elasticsearch

## Objectifs

Ce projet vous permettra de :
- Vérifier le bon fonctionnement de la stack SIEM
- Analyser les données indexées par Filebeat (Suricata)
- Détecter des anomalies sans ML
- Détecter des anomalies avec ML (Isolation Forest)

## Modalité
- groupe de 3 à 4
- output : projet git avec ce notebook détaillé et complété

## Architecture SIEM

```
Suricata   →   Filebeat   →   Elasticsearch   →   Kibana
(IDS/HIDS)   (Collecteur)      (Stockage)   (Visualisation/Alerting)
```

## Prérequis

1. Démarrer la stack :
```bash
docker-compose -f docker-compose-siem.yml up -d
```

2. Attendre quelques minutes que Suricata génère des logs et que Filebeat les indexe dans Elasticsearch.

SyntaxError: invalid character '→' (U+2192) (3498685080.py, line 18)

In [2]:
# Configuration et connexion à Elasticsearch
from elasticsearch import Elasticsearch
from datetime import datetime, timedelta
import json
import subprocess
import os
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter("ignore", InsecureRequestWarning)
# Configuration
ES_HOST = "https://localhost:9200"
ES_USER = "elastic"
ES_PASSWORD = "changeme"  # Modifiez selon votre .env

# Connexion
es = Elasticsearch(
    [ES_HOST],
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False
)

# Vérification de la connexion
health = es.cluster.health()
print(f"✅ Cluster Elasticsearch: {health['status']} ({health['number_of_nodes']} nœuds)")

✅ Cluster Elasticsearch: green (3 nœuds)


/Users/auteqia/Dev/elasticsearch_cookbook/.venv/lib/python3.13/site-packages/elasticsearch/_sync/client/__init__.py:313: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


## 1. Vérification de la stack

In [3]:
# Vérification des services Docker
services = ['es01', 'es02', 'es03', 'kibana', 'suricata', 'filebeat']
running = []

for service in services:
    try:
        result = subprocess.run(
            ['docker', 'ps', '--filter', f'name={service}', '--format', '{{.Names}}'],
            capture_output=True, text=True, timeout=5
        )
        if service in result.stdout:
            running.append(service)
            print(f"✅ {service}")
        else:
            print(f"❌ {service}")
    except:
        print(f"❌ {service}")

if len(running) == len(services):
    print(f"\n✅ Tous les services sont démarrés ({len(running)}/{len(services)})")
else:
    print(f"\n⚠️  Services démarrés: {len(running)}/{len(services)}")

✅ es01
✅ es02
✅ es03
✅ kibana
✅ suricata
✅ filebeat

✅ Tous les services sont démarrés (6/6)


## 2. Vérification de l'injection des données

In [6]:
# Recherche des index Suricata
def get_suricata_index():
    """Retourne le nom de l'index Suricata le plus récent"""
    try:
        indices = es.indices.get(index="suricata-*")
        if indices:
            return sorted(indices.keys())[-1]
    except:
        pass
    return "suricata-*"

index_name = get_suricata_index()

# Comptage des documents
try:
    count = es.count(index=index_name)
    print(f"📊 Index: {index_name}")
    print(f"📈 Nombre de documents: {count['count']:,}")
    
    # Exemple de document
    if count['count'] > 0:
        sample = es.search(index=index_name, size=1, query={"match_all": {}})
        if sample['hits']['hits']:
            doc = sample['hits']['hits'][0]['_source']
            print(f"\n📄 Exemple de document:")
            print(f"   Type: {doc.get('event_type', 'N/A')}")
            print(f"   Timestamp: {doc.get('@timestamp', doc.get('timestamp', 'N/A'))}")
            if 'src_ip' in doc:
                print(f"   Source: {doc.get('src_ip')}:{doc.get('src_port', 'N/A')}")
                print(f"   Destination: {doc.get('dest_ip')}:{doc.get('dest_port', 'N/A')}")
            if 'alert' in doc:
                alert = doc['alert']
                print(f"   Alerte: {alert.get('signature', 'N/A')}")
                print(f"   Sévérité: {alert.get('severity', 'N/A')}")
except Exception as e:
    print(f"❌ Erreur: {e}")

📊 Index: .ds-suricata-2026.03.02-2026.03.02-000001
📈 Nombre de documents: 4,371

📄 Exemple de document:
   Type: dns
   Timestamp: 2026-03-02T15:01:12.692Z
   Source: 192.168.139.2:38153
   Destination: 0.250.250.200:53


## 4. Simuler des comportements anormaux


#### Simulation de scans suspects (en ligne de commande ou en python) 

In [5]:
import socket

def simulate_port_scan(target_ip="192.168.65.3", max_port=3000):
    print(f"scan de ports sur {target_ip}")
    for port in range(1, max_port):
        try:
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.settimeout(0.01)
            s.connect((target_ip, port))
            s.close()
        except:
            pass
    print("scan terminé")

simulate_port_scan()

scan de ports sur 192.168.65.3
scan terminé


#### Simulation de burst HTTP en ligne de commande (en ligne de commande ou en python) 

In [7]:
import requests
import time

def simulate_http_burst(target_url="http://192.168.65.3:80", num_requests=500):
    print(f"Lancement du burst HTTP vers {target_url}...")
    for i in range(num_requests):
        try:
            requests.get(target_url, timeout=0.5)
        except requests.exceptions.RequestException:
            pass
    print(f" Burst de {num_requests} requêtes terminé")

simulate_http_burst()

Lancement du burst HTTP vers http://192.168.65.3:80...
 Burst de 500 requêtes terminé


#### Simulation de [à continuer]

In [ ]:
import socket
import time

def simulate_mysql_connections(target_ip="192.168.65.3", target_port=3306, num_attempts=50):
    print(f"🚀 Lancement de la simulation vers MySQL ({target_ip}:{target_port})...")
    
    success_count = 0
    fail_count = 0
    
    for i in range(num_attempts):
        try:
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.settimeout(0.5) 
            
            s.connect((target_ip, target_port))
            
            payload = f"USER admin\nSELECT * FROM users WHERE id={i} OR '1'='1';\n".encode()
            s.sendall(payload)
            s.close()
            
            success_count += 1
            time.sleep(0.1) 
        except Exception:
            fail_count += 1
            
    print(f"Simulation terminée.")
    print(f"Bilan : {success_count} payloads envoyés, {fail_count} connexions refusées (normal si le port n'est pas ouvert).")


simulate_mysql_connections()

## 4. Détection d'anomalies sans ML

## 4. Détection d'anomalie (sans Machine Learning)

L'objectif est de construire des règles qui détectent vos simulations de comportements précédents.

Un exemple de détection d'anomalies basiques basées sur des règles:
```python
def detect_anomalies_basic():
    """Détecte des anomalies sans ML"""
    anomalies = []
    
    # 1. IPs sources avec beaucoup d'alertes différentes (scan suspect)
    query = {
        "size": 0,
        "query": {"term": {"event_type": "alert"}},
        "aggs": {
            "suspicious_ips": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "unique_signatures": {"cardinality": {"field": "alert.signature_id"}},
                    "unique_dest_ips": {"cardinality": {"field": "dest_ip"}},
                    "total_alerts": {"value_count": {"field": "event_type"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query)
    
    print("Détection d'anomalies (règles basiques)\n")

    for bucket in result['aggregations']['suspicious_ips']['buckets']:
        ip = bucket['key']
        unique_sigs = bucket['unique_signatures']['value']
        unique_dests = bucket['unique_dest_ips']['value']
        total = bucket['total_alerts']['value']
        
        # Critères d'anomalie
        if unique_sigs > 3 or unique_dests > 5:
            anomalies.append({
                "ip": ip,
                "type": "Scan suspect",
                "signatures": unique_sigs,
                "destinations": unique_dests,
                "total": total
            })
            print(f"{ip}: {unique_sigs} signatures, {unique_dests} destinations ({total} alertes)")

    return anomalies

anomalies = detect_anomalies_basic()
if not anomalies:
    print("\nAucune anomalie détectée avec les règles basiques")

detect_anomalies_basic()
```

In [10]:
def detect_http_burst():
    """Détecte les bursts HTTP en comptant les événements par IP sur 1 minute"""
    query = {
        "size": 0,
        "query": {
            "bool": {
                "must": [{"term": {"event_type": "flow"}}] 
            }
        },
        "aggs": {
            "ips_suspectes": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "requetes_par_minute": {
                        "date_histogram": {
                            "field": "@timestamp",
                            "fixed_interval": "1m"
                        }
                    }
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query)
    print("\nDétection de Burst HTTP (Règles basiques)")
    
    for bucket in result['aggregations']['ips_suspectes']['buckets']:
        ip = bucket['key']
        for time_bucket in bucket['requetes_par_minute']['buckets']:
            count = time_bucket['doc_count']
            if count > 100: 
                print(f"Anomalie de volume : l'IP {ip} a envoyé {count} requêtes en 1 minute à {time_bucket['key_as_string']}")

detect_http_burst()


Détection de Burst HTTP (Règles basiques)
Anomalie de volume : l'IP fd07:b51a:cc66:00f0:0000:0000:0000:0001 a envoyé 495 requêtes en 1 minute à 2026-03-02T14:44:00.000Z
Anomalie de volume : l'IP fd07:b51a:cc66:00f0:0000:0000:0000:0001 a envoyé 503 requêtes en 1 minute à 2026-03-02T14:46:00.000Z
Anomalie de volume : l'IP fd07:b51a:cc66:00f0:0000:0000:0000:0001 a envoyé 480 requêtes en 1 minute à 2026-03-02T14:47:00.000Z
Anomalie de volume : l'IP fd07:b51a:cc66:00f0:0000:0000:0000:0001 a envoyé 517 requêtes en 1 minute à 2026-03-02T14:48:00.000Z


## 5. Détection d'anomalies avec ML

L'objectif est d'à partir de données labelisées comme étant anomarles ou non, construire un jeu de donnée et entrainer un algorithme de machine learning (classifier binaire) sur ces données. Le modèle devra ensuite etre appelé pour prédire les futurs comportements anormaux.

Indice: le modèle Isolation Forest pourrait etre utile

In [13]:
import pandas as pd
from sklearn.ensemble import IsolationForest

def train_and_detect_ml():
    # 1. Requête corrigée (suppression de l'agrégation sur _id)
    query = {
        "size": 0,
        "aggs": {
            "by_ip": {
                "terms": {"field": "src_ip", "size": 1000},
                "aggs": {
                    "unique_dest_ports": {"cardinality": {"field": "dest_port"}},
                    "unique_signatures": {"cardinality": {"field": "alert.signature_id"}}
                }
            }
        }
    }
    
    res = es.search(index=index_name, body=query)
    
    data = []
    ips = []
    
    # 2. Utilisation de la clé native 'doc_count'
    for bucket in res['aggregations']['by_ip']['buckets']:
        ips.append(bucket['key'])
        data.append({
            "total_events": bucket['doc_count'],  # <-- Le changement est ici !
            "unique_dest_ports": bucket['unique_dest_ports']['value'],
            "unique_signatures": bucket['unique_signatures']['value']
        })
        
    df = pd.DataFrame(data)
    
    if df.empty:
        print("Pas assez de données pour le ML.")
        return
        
    # contamination = 0.05 signifie qu'on estime qu'environ 5% du trafic est malveillant
    model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    
    # Le modèle prédit : 1 (Normal) ou -1 (Anomalie)
    df['anomaly'] = model.fit_predict(df[['total_events', 'unique_dest_ports', 'unique_signatures']])
    df['ip'] = ips
    
    anomalies = df[df['anomaly'] == -1]
    
    print(f"\n🧠 Détection Machine Learning (Isolation Forest) terminée sur {len(df)} IPs.")
    if not anomalies.empty:
        print("⚠️ Anomalies détectées par l'algorithme :")
        print(anomalies[['ip', 'total_events', 'unique_dest_ports', 'unique_signatures']])
    else:
        print("✅ Aucune anomalie détectée par le ML.")

# Lancement de la fonction
train_and_detect_ml()


🧠 Détection Machine Learning (Isolation Forest) terminée sur 8 IPs.
⚠️ Anomalies détectées par l'algorithme :
              ip  total_events  unique_dest_ports  unique_signatures
3  0.250.250.200            39                 39                  0


## Bonus
- améliorer la stack (ex: ajout de Wazuh)
- dashboard Kibana pour voir en live les simulations de comportements anormaux et les détection d'anomalie (timeline des alertes, etc.)
- analyse statistique avancée
- simulations de comportement anormaux avancée
- utilisation du module de détection d'anomalie d'Elasticsearch (https://www.elastic.co/docs/explore-analyze/machine-learning/anomaly-detection)
- collaboration en groupe sur le projet Git (Pull requests, commits, etc.)
- utilisation de docker / docker compose / devcontainer
- etc.